In [1]:
import torch
import numpy as np


from dinosaw.wrappers import ModelTypes, MODEL_NAMES, get_models
from dinosaw.utils import do_2D_pca, get_features, add_custom_font

from dinosaw.utils import do_2D_pca
from skimage.transform import resize
from skimage.exposure import equalize_adapthist

from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import Literal, TypeAlias

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


selected_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dvt_dinov2_s', 'alibi_coco_dinov2_s', 'dinov3_s+', 'alibi_dinov3_s+_norm_wrap_ms')
models = get_models(selected_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")


n_dims = 384

2026-07-24 13:25:00 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-24 13:25:00 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-07-24 13:25:01 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 13:25:01 | I | factory.py                 : 152 | Building wrapper 'dvt_dinov2_s' on device cuda:0
2026-07-24 13:25:01 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-07-24 13:25:01 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 13:25:01 | I | factory.py                 : 152 | Building wrapper 'alibi_coco_dinov2_s' on

In [3]:
image_names = ['labradors', 'new_york', 'bimodal', 'biphase_steel_crop']

shortest_side = 683
longest_side = 1024

images = []

for image_name in image_names:
    image = Image.open(f'data/more_pcas/{image_name}.png')
    ih, iw = image.size[::-1]
    scale = shortest_side / min(ih, iw)
    image = image.resize((int(iw * scale), int(ih * scale)))
    
    ox, oy = (image.size[0] - longest_side) // 2, (image.size[1] - shortest_side) // 2
    image = image.crop((ox, oy, ox + longest_side, oy + shortest_side))

    if image_name == 'biphase_steel_crop':
        image = equalize_adapthist(np.array(image), clip_limit=0.01)
        image = Image.fromarray((image * 255).astype(np.uint8))

    images.append(image)


In [4]:
features: dict[ModelTypes, list[np.ndarray]] = {key: [] for key in selected_models}

pca_scaling = 'std'
for key, model in models.items():
    for image in images:
        feats = get_features(model, image)
        reduced = do_2D_pca(feats, 3, pre_norm=pca_scaling, post_norm='minmax')[:, :, 0:9]
        features[key].append(reduced)

2026-07-24 13:25:03 | I | wrapper.py                 :  92 | Processing image, size: [1024, 683]
2026-07-24 13:25:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,672,1022] -> f: [1,384,48,73]
2026-07-24 13:25:03 | I | wrapper.py                 :  92 | Processing image, size: [1024, 683]
2026-07-24 13:25:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,672,1022] -> f: [1,384,48,73]
2026-07-24 13:25:03 | I | wrapper.py                 :  92 | Processing image, size: [1024, 683]
2026-07-24 13:25:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,672,1022] -> f: [1,384,48,73]
2026-07-24 13:25:03 | I | wrapper.py                 :  92 | Processing image, size: [1024, 683]
2026-07-24 13:25:03 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,672,1022] -> f: [1,384,48,73]
2026-07-24 13:25:03 | I | wrapper.py                 :  92 | Processing image, size: [1024, 683]
2026-07-24 13:25:03 | I | wrapper.py           

In [5]:
def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [7]:
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')
n_rows, n_cols = len(selected_models) + 1, len(images)
W, H = 7.5, 2.3 * 3

titles = ["Labradors", "New York", "Bimodal cathode", "Biphase steel"]
fig, axs = plt.subplots(n_rows, n_cols, figsize=(W, H))

for col, image in enumerate(images):
    axs[0, col].imshow(image, rasterized=True)
    hide_axes(axs[0, col])
    axs[0, col].set_title(titles[col], weight=500)

    for row, key in enumerate(selected_models):
        ax = axs[row + 1, col]
        ax.imshow(features[key][col], rasterized=True)
        hide_axes(ax)

        if col == 0:
            weight = 700 if 'alibi' in key else 500
            name = MODEL_NAMES[key]
            name = name.replace('(COCO)', '')
            ax.set_ylabel(name, weight=weight)

scaling = "None" if pca_scaling is None else pca_scaling
plt.suptitle(f"Pre-PCA scaling: {scaling}")

SAVE = True
if SAVE:
    plt.savefig("saved/S12.pdf", dpi=300, bbox_inches='tight')
    plt.close()


findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight 500, now using 300.
findfont: Failed to find font weight normal, now using 300.
